#### Data Extraction and selection

In [5]:
import pandas as pd 
import random  # random sampling 
import os       # File path operations 
import glob     # Find files in folder 


In [7]:
# 1. Path to your unzipped first folder
folder_path = '../CSV-01-12/01-12/' 
all_files = glob.glob(os.path.join(folder_path, "*.csv"))
all_files

['../CSV-01-12/01-12\\DrDoS_DNS.csv',
 '../CSV-01-12/01-12\\DrDoS_LDAP.csv',
 '../CSV-01-12/01-12\\DrDoS_MSSQL.csv',
 '../CSV-01-12/01-12\\DrDoS_NetBIOS.csv',
 '../CSV-01-12/01-12\\DrDoS_NTP.csv',
 '../CSV-01-12/01-12\\DrDoS_SNMP.csv',
 '../CSV-01-12/01-12\\DrDoS_SSDP.csv',
 '../CSV-01-12/01-12\\DrDoS_UDP.csv',
 '../CSV-01-12/01-12\\Syn.csv',
 '../CSV-01-12/01-12\\TFTP.csv',
 '../CSV-01-12/01-12\\UDPLag.csv']

In [8]:
sampled_dataframes = []
TARGET_ROWS_PER_FILE = 60000

print("Starting memory-efficient random sampling...")

for file in all_files:
    file_name = os.path.basename(file)
    print(f"Processing {file_name}...")
    
    # Step A: Quickly count total rows in the file (without loading it into memory)
    with open(file, 'r', encoding='utf-8') as f:
        total_rows = sum(1 for _ in f)
    
    # Step B: Calculate the probability needed to get roughly 50,000 rows
    # If the file has 5,000,000 rows, we only want 1% of them.
    keep_probability = TARGET_ROWS_PER_FILE / total_rows
    
    # Step C: Load the CSV, randomly skipping rows based on our probability
    # We always keep row 0 because it contains the column names
    df_sample = pd.read_csv(
        file, 
        skiprows=lambda x: x > 0 and random.random() > keep_probability,
        low_memory=False
    )
    
    # Clean up column names (strip remove leading and trailing spaces)
    df_sample.columns = df_sample.columns.str.strip()
    sampled_dataframes.append(df_sample)
    print(f" -> Kept {len(df_sample)} random rows from {file_name}")

Starting memory-efficient random sampling...
Processing DrDoS_DNS.csv...
 -> Kept 59877 random rows from DrDoS_DNS.csv
Processing DrDoS_LDAP.csv...
 -> Kept 60253 random rows from DrDoS_LDAP.csv
Processing DrDoS_MSSQL.csv...
 -> Kept 60283 random rows from DrDoS_MSSQL.csv
Processing DrDoS_NetBIOS.csv...
 -> Kept 59689 random rows from DrDoS_NetBIOS.csv
Processing DrDoS_NTP.csv...
 -> Kept 59910 random rows from DrDoS_NTP.csv
Processing DrDoS_SNMP.csv...
 -> Kept 59758 random rows from DrDoS_SNMP.csv
Processing DrDoS_SSDP.csv...
 -> Kept 59702 random rows from DrDoS_SSDP.csv
Processing DrDoS_UDP.csv...
 -> Kept 59794 random rows from DrDoS_UDP.csv
Processing Syn.csv...
 -> Kept 59805 random rows from Syn.csv
Processing TFTP.csv...
 -> Kept 60334 random rows from TFTP.csv
Processing UDPLag.csv...
 -> Kept 60050 random rows from UDPLag.csv


In [9]:
# 2. Merge all the randomly sampled chunks into one dataset
print("Merging datasets...")
main_train_dataset = pd.concat(sampled_dataframes, ignore_index=True)

# Save your beautiful, balanced, randomly sampled training data!
main_train_dataset.to_csv("CICDDOS_Train_Data.csv", index=False)
print(f"Done! Final training dataset size: {len(main_train_dataset)} rows.")

Merging datasets...
Done! Final training dataset size: 659455 rows.


In [ ]:
# 3. The 'Honesty' Cleaning Step (Handling Inf and NaN)
# import numpy as np
# main_train_dataset.replace([np.inf, -np.inf], np.nan, inplace=True)
# main_train_dataset.dropna(inplace=True)
